# Analysing forest recovery after logging events - comparison with satellite data

During the previous practical, you learned about the data that the Victorian Department of Energy, Environment and Climate Action keep on the management of logging coupes over time, and exported all the coupes that can be easily compared with satellite data from the Sentinel-2 constellation.

Your team has reviewed your file and selected three known logging coupes, which they would like you to investigate. Each coupe uses a different type of silvicultural system for harvesting, and your team are curious about how these events show up in satellite data. 

In this practical, you'll load and view the satellite data associated with each logging event, both as an image, and using the normalised difference vegetation index (NDVI). You'll then view the time series of how NDVI is changing over time for each event.

## Overview

During this activity, you will learn to

* select specific rows from a table of geospatial data
* load satellite data for the time and location corresponding to a given logging event
* review satellite data, both as images and as timeseries

Most of the Python code for loading and analysing satellite data has been provided for you, but there will be a number of opportunities to write your own code, as well as customise existing code to explore different results. The focus of this session is to review the results and think about what you can learn from satellite data.

### Guiding text

This practical contains a number of headings to help guide you. 

* <span style="color:blue;font-weight:bold">Your task</span>: This indicates there is a task you must complete before proceeding. It will usually require you to add code or text before you can move on.
* <span style="color:green;font-weight:bold">Need some help?</span>: Your demonstrators are here to help -- this text is there to remind you to ask for help if you're not sure what you need to do. You can ask for help at any time.
* <span style="color:orange;font-weight:bold">Going further</span>: This indicates that there is an *optional* extension you can try if you've already completed the tasks.
* **Code explanation**: The text following this header will provide you with more information about how the code works -- you only need to read this if you're interested.

### Errors and warnings

It is normal when developing and running Python code to encounter errors and warnings. If you see a red box containing text appear, it is an error or a warning, which is the computer's way of giving you feedback that something isn't quite right. Read the [common errors guide](error_guide.ipynb) to learn more about what might be causing the issue, and then try and resolve it on your own, or with help from your demonstrator.

### Terminology

In the previous practical, your colleagues provided some useful definitions:

* [Silviculture](https://www.forestrycorporation.com.au/operations/silviculture) is the science of forestry.
* A [coupe](https://www.vicforests.com.au/vicforest-forest-management/ops-planning/where-vicforests-operates/timber-release-plan) is a defined area in a forest that timber can be harvested from.
* A [silvicultural system](https://www.fs.usda.gov/Internet/FSE_DOCUMENTS/fseprd530429.pdf) is the planned strategy for managing a coupe, including the harvesting and regeneration of timber.

## Notebook setup

In addition to `pandas` and `geopandas`, you'll now also need a few extra packages:
* `pystac.client` - allows you to connect to Digital Earth Australia's online data catalog
* `odc.stac` - allows you to load data from Digital Earth Australia's online data catalog
* `odc.geo` - provides extra geometry utility functions that help when loading geospatial data
* `matplotlib.pyplot` - provides useful utility functions for making plots

The other analyst on your team has also provided a number of extra functions that you'll use throughout the notebook.
One function is to help you make plots that compare true-colour imagery to NDVI values for a given set of images.

To run the code, click on the next cell, and press `Shift`+`Enter` on your keyboard.

In [ ]:
# import key packages for data handling
import geopandas
import pandas

# import key packages for finding and loading satellite images
from pystac.client import Client
from odc.stac import configure_s3_access, load
import odc.geo.xr
from odc.geo.geom import Geometry
from odc.geo.crs import CRS

# import key plotting packages
import matplotlib.pyplot as plt

# import custom plotting functions from your colleague
from plotting_functions import plot_rgb_ndvi

# Change a pandas setting to view all columns and all rows of loaded data
pandas.set_option("display.max_columns", None)
pandas.set_option("display.max_rows", None)

## Load the data

In the last practical, you exported a file called "LOG_SEASON_FILTERED.gpkg". This contains all the logging events that can be matched with Sentinel-2 data. The **path** to the data is `"LOG_SEASON_FILTERED.gpkg"`.

From the first practical, recall that you can use the `read_file` function from GeoPandas to load the data. You can read more about this function in the [geopandas documentation](https://geopandas.org/en/stable/docs/reference/api/geopandas.read_file.html#geopandas.read_file).

### <span style="color:blue;font-weight:bold">Your task</span>

> Load the logging data using the `read_file` function from `geopandas` and assign the loaded data to a variable called `logging_season_data`. Then, use the empty cell below it to view the **first 5 rows** of the data.
>
> What's provided:
> * The variable you'll assign the data to: `logging_season_data`.
> * The `=` sign that will assign the results of any code that comes after it to the variable.
> * An empty cell you can use to view the first 5 rows of the data.
>
> What you'll need to add:
> * After the `=` sign, type `geopandas.read_file()` to call the function.
> * Inside the `()` for the function, type the path to the file: `"LOG_SEASON_FILTERED.gpkg"`
> * In the empty cell, type the geopandas function for viewing the first 5 rows of the data. If you're not sure what the function is, open the previous practical and review the second task.
>
>After adding the required code to the cell below, run the cells by clicking on each cell and pressing `Shift`+`Enter` on your keyboard.

### <span style="color:green;font-weight:bold">Need some help?</span>
> If you see a <span style="color:red">NameError</span>, make sure you have included double quotes (`" "`) around the name of the file.
> 
>If you're not sure what to do, get in touch with a demonstrator (in the room or online) and show them your screen to talk through what you've tried and what the next step might be.

In [ ]:
logging_season_data =

In [ ]:
# View the first five rows using the head() method


## Select logging coupes for analysis

Your team has identified three logging coupes for you to investigate, with each coupe demonstrating a different silvicultural system. They have provided you with the event LOGHISTID values so that you can select them from the filtered data you created during the previous practical:

* **Clearfelling system**: "14/770/507/0011/201920/00"
* **Regrowth retention harvesting system**: "08/286/505/0029/201920/00"
* **Variable retention 1 system**: "16/686/510/0026/201920/00"

Your colleague has provided these  LOGHISTID values as a Python [dictionary](https://www.w3schools.com/python/python_dictionaries.asp), and has provided some code to only select the rows in your dataset with these LOGHISTID values. Run the cells below, and then complete the exercise.

In [ ]:
# Store the events of interest in a Python dictionary
events_of_interest = {
    "Clearfelling": "14/770/507/0011/201920/00",
    "Regrowth retention harvesting": "08/286/505/0029/201920/00",
    "Variable retention 1": "16/686/510/0026/201920/00",
}

# Get a list containing the IDs from the dictionary
event_ids = list(events_of_interest.values())

# Identify the rows that correspond to the IDs in the list
rows_of_interest = logging_season_data.loc[:, "LOGHISTID"].isin(event_ids)

# Make a new table only containing the rows of interest
logging_events_to_analyse = logging_season_data.loc[rows_of_interest, :].reset_index(drop=True)

# View the new table
logging_events_to_analyse

**Code explanation**

> The above code uses two `GeoDataFrame` methods: `loc[]` and `isin()`. You can review all methods in the [GeoPandas documentation](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.html).
> 
> * `loc[row, column]` selects the data from the provided `row` and `column` values. Providing a value of `:` selects everything
> * `isin(list)` produces a value of `True` or `False` for each row depending on whether the row contains text that appears in the `list`.

### Exercise: Reviewing the events

Your team is interested in the answers to the following questions:

* **Question 1**: What is the **earliest start date** across the three events?
* **Question 2**: What is the **latest end date** across the three events?
* **Question 3**: Which **forest type** was harvested in all three events?

### <span style="color:blue;font-weight:bold">Your task</span>
> Review the table of selected events above, and answer the questions below.
>
>Double click the text below to add your answers.

### Your answers

**Question 1**: 

**Question 2**:

**Question 3**:

### Extract geometries

Each logging event has a geospatial geometry that maps out the total extent of the logging event and where it took place.
Your colleague has provided some code to extract the geometry for each event, and store it in a Python dictionary so that you can use it again later.

In [ ]:
# Get the coordinate reference system from the geopandas GeoDataFrame
crs = CRS(logging_events_to_analyse.crs)

# Create an empty dictionary to store the event geometries
event_geometries = {}

# Loop through the GeoDataFrame and extract 
for row_number, row_data in logging_events_to_analyse.iterrows():
    # Convert the row geometry to an odc.geo geometry type
    event_geometry = Geometry(row_data.geometry, crs)

    # Store the geometry for the event (as described by its Silvicultural System name) in the dictionary
    event_type = row_data["X_SILVSYS"]
    event_geometries[event_type] = event_geometry

**Code explanation**

> The above code uses two classes from the `odc.geo` library:
> * `CRS` - this standardises the coordinate reference system information from the GeoDataFrame, making it easy to use.
> * `Geometry` - this standardises the geometry from the GeoDataFrame, making it easy to use. It requires a coordinate reference system to ensure it's correctly located on Earth.
>
> The code then uses the `GeoDataFrame.iterrows()` method to loop through each row in the `logging_events_to_analyse` GeoDataFrame.

### Display geometries

When working with the `odc.geo` Geometry class, it's possible to display the shape of the geometry. 
Doing this helps you see what real logging events look like spatially.

In [ ]:
for event_type, event_geom in event_geometries.items():
    print(event_type)
    display(event_geom)

## Loading satellite data for each event

### Connecting to Digital Earth Australia's Spatio-Temporal Asset Catalog (STAC)

Digital Earth Australia's data is stored in a Spatio-Temporal Asset Catalog, allowing you to query and load data over the internet. 
The first step is to connect to the catalog, which is achieved using the code below.

In [ ]:
# The catalog URL for Digital Earth Australia's STAC API
catalog = "https://explorer.dea.ga.gov.au/stac"

# Pystac_client's Client class is used to connect to the catalog
stac_client = Client.open(catalog)

# Configure settings for reading from Digital Earth Australia's STAC
configure_s3_access(
    cloud_defaults=True,
    aws_unsigned=True,
)

**Code explanation**

> * The first step specifies the URL for Digital Earth Australia's Spatio-Temporal Asset Catalog.
> * The second step opens the catalog using the `Client` class from the `pystac.client` library.
> * The final step applies a set of configuration settings that allow you to read satellite imagery from Digital Earth Australia's Amazon Web Services (AWS) data collection

### Constructing the query

In the next cell, your colleague has provided you with some recommended settings for finding and loading relevant satellite images. The recommended settings are as follows:

#### Search settings
* **start_date** and **end_date**: The start and end date to search for data over. Your colleague suggests looking at events from a bit before and after the actual logging events.
* **collections_to_search**: the satellite data to load from. `"ga_s2am_ard_3"` is the analysis-ready data product for Sentinel-2A, and `"ga_s2bm_ard_3"` is the analysis-ready data product for Sentinel-2B.
* **minimum_clear_percentage**: a threshold for selecting clear images. If the percentage of clear pixels in the image is greater than this threshold, it will be included in the load. If the percentage of clear pixels in the image is smaller than this threshold, it will not be included in the load. This helps us select images that are largely cloud-free.

#### Load settings
* **bands_to_load**: the Sentinel-2 bands to load. The `nbart_` prefix has to do with how the data have been processed. For visual interpretation, you need the red, green and blue bands. For calculating NDVI, you need the red and near-infrared (`nir_1`) bands.
* **output_resolution**: the resolution (in metres) for each pixel in the image. The first number (`10`) specifies 10m in the horizonal direction, and the second number (`-10`) specifies 10m in the vertical direction.
* **output_crs**: the coordinate reference system (CRS) to use for the loaded data. [EPSG:3577](https://epsg.io/3577) is the Australian Albers equal-area projection.

In [ ]:
# Search parameters
start_date = "2019-06-01"
end_date = "2021-02-28"
collections_to_search = ["ga_s2am_ard_3", "ga_s2bm_ard_3"]
minimum_clear_percentage = 80

# Load parameters
bands_to_load = ["nbart_red", "nbart_green", "nbart_blue", "nbart_nir_1"]
output_resolution = 10
output_crs = "EPSG:3577"

### Searching for images
When working with Digital Earth Australia's STAC catalog, the first step is to identify all images that match the query parameters.
This is done by using a Python [for loop](https://www.w3schools.com/python/python_for_loops.asp), which loads data for each event in turn, and stores the outputs.

The loop will run over the dictionary of geometries created earlier, creating a new dictionary containing the images found for each event.

In [ ]:
event_images = {}

for event_type, event_geom in event_geometries.items():
    print(f"Finding images for the {event_type} event")

    # Set the query, specifying the start and end date, the collections, the geometry to return data for, and the minimum clear percentage
    images = stac_client.search(
        datetime=f"{start_date}/{end_date}",
        intersects=event_geom,
        collections=collections_to_search,
        filter={"op": ">=", "args": [{"property": "fmask:clear"}, minimum_clear_percentage]}
    ).item_collection()

    print(f"    Found: {len(images):d} images")
    event_images[event_type] = images

### Loading data

The next cell contains multiple steps that are used to load the data for each event. This is done by using a Python [for loop](https://www.w3schools.com/python/python_for_loops.asp), which loads data for each event in turn, and stores the outputs.

> **The data loading step will take 3-5 minutes!** 

Please be patient and keep an eye on the output. While you are waiting, you can read more about the steps involved. The steps are described below the next cell. You will know the code has finished running when you see the message **"All data loading is complete! You can progress to the next step."** in the output.

Run the code cell below, then scroll down to read about the steps while the data loads.

In [ ]:
# Create empty dictionary to store results in
event_data = {}

for event_type, event_image_list in event_images.items():

    # Load all images associated with the harvesting event
    print(f"Loading images for {event_type} event")
    event_ds = load(
        items=event_image_list,
        bands=bands_to_load,
        crs=output_crs,
        resolution=output_resolution,
        groupby="solar_day",
        geopolygon=event_geometries[event_type]
    )

    # Remove all pixels outside the event geometry, so that we only analyse pixels affected by the harvesting event
    print(f"   Masking images for {event_type} event, keeping pixels within the event geometry")
    event_ds_masked = event_ds.odc.mask(event_geometries[event_type])

    # Calculate NDVI using (NIR - Red)/(NIR + Red)
    print(f"   Calculating NDVI for {event_type} event")
    event_ds_masked["NDVI"] = (event_ds_masked.nbart_nir_1 - event_ds_masked.nbart_red) / (event_ds_masked.nbart_nir_1 + event_ds_masked.nbart_red)

    # Group by seasonal intervals and calculate a median
    print(f"   Calculating seasonal median for {event_type} event")
    event_ds_median = event_ds_masked.resample(time="QS-JUN").median()

    # Store the results in the event_data dictionary
    event_data[event_type] = event_ds_median

print("All data loading is complete! You can progress to the next step.")

**Code Explanation**

The Python [for loop](https://www.w3schools.com/python/python_for_loops.asp) allows us to repeat the same set of actions for each event. The actions are:

> **1. Load all the found images for the event.** The load command includes the bands to load, the output coordinate reference system (CRS) to use, the resolution in metres, and the geometry to load over.
> 
> **2. Apply a geometry mask to only keep data associated with the clearing event.** This step uses a [special masking function](https://odc-geo.readthedocs.io/en/latest/_api/odc.geo.xr.ODCExtension.mask.html#odc.geo.xr.ODCExtension.mask) that is available for data loaded using the `odc.stac` package.
> 
> **3. Group by three-month seasonal intervals and calculate the median for each band.** This step allows us to create a representative dataset over a seasonal period by selecting the median pixel value from all values loaded for that period. The median is valuable because it tends to select non-cloudy values for each pixel. 
> 
> For each median, the date listed in the dataset will be the **start of the seasonal window**. For example, the seasonal windows and listed median dates for the first year of this analysis are
> 
> | season | seasonal date range | listed median date |  
> |--------|---------------------|--------------------|  
> | Winter | Jun 2019 – Aug 2019 | 2019-06-01         |  
> | Spring | Sep 2019 – Nov 2019 | 2019-09-01         |  
> | Summer | Dec 2019 – Feb 2020 | 2019-12-01         |  
> | Autumn | Mar 2020 – May 2020 | 2020-03-01         |  
> 
> **4. Calculate NDVI using (NIR - Red)/(NIR + Red)**. This step takes the loaded red and near-infrared bands from the composite and calculates the corresponding NDVI values. NDVI is a satellite band index that indicates the presence of vegetation, with values ranging from -1 to 1. Higher values typically correspond to dense, green vegetation.
> 
> **5. Store the results in the event_data dictionary**. This step allows us to store the data for each event and use it for our analysis.

## Visualising loaded data

### Spatial time series
Writing code to plot data in Python can be time-consuming to develop, so your colleague has dug out an old function they once made to help you. The function is called `plot_rgb_ndvi()` and it takes two arguments: the first is the event data and the second is the name of the event type to use as a title. It will show the visual image of the logging area (the combination of red, green and blue bands) on the left-hand side, and the corresponding NDVI values on the right hand side. It will plot each composite that was generated, one after the other.

Run the following cell to view the RGB and NDVI images for each logging event.

### <span style="color:orange;font-weight:bold">Going further</span>
> This is an optional exercise to further your own understanding. There are no questions to answer for this component.
> 
> **What are the different steps involved in creating these visualisations?** Review the plotting function created by your colleague in the [plotting_functions.py file](./plotting_functions.py)

In [ ]:
# Plot RGB and NDVI for the event areas
for event_type, event_ds in event_data.items():
    plot_rgb_ndvi(event_ds, event_type)

### Summary time series

Your colleague has provided additional code to calculate and plot the average NDVI value for each composite, which will allow you to see the general trend in the presence of vegetation for the three events.

Run the cell below to view the average NDVI over time.

In [ ]:
# Store the plot labels to add at the end
labels = []

# Create the figure
fig, ax = plt.subplots(figsize=(12, 6))

# Add a title
fig.suptitle("Change in NDVI over time for different logging events", fontsize=16)

# Plot the mean (average) NDVI for each event
for event_type, event_ds in event_data.items():
    event_ds.NDVI.mean(dim=["x", "y"]).plot(marker="o", linestyle="--", add_legend=False, ax=ax)
    labels.append(event_type)


# Add the labels and display the plot
plt.legend(labels, ncol=1, fontsize=12)
plt.show()

## Analysis

Congratulations! You have successfully loaded and visualised satellite data for the three logging events! Now, your colleagues are curious to know what you found.

Your team ask you to report back on the following:

* **Question 1**: In 1-2 sentences, describe **one similarity between the three different events** that you noticed when looking at the RGB and NDVI spatial time series.
* **Question 2**: In 1-2 sentences, describe **one difference between the three different events** that you noticed when looking at the RGB and NDVI spatial time series.
* **Question 3**: When looking at the summary time series for the **Clearfelling event**, during which seasonal period and year would you say the Clearfelling began? Is this consistent with the start date in the event table you created when [selecting logging coupes for analysis](#Select-logging-coupes-for-analysis)? Why or why not?

### <span style="color:blue;font-weight:bold">Your task</span>
> Review the spatial and summary time series, and answer the analysis questions below.
> Double click the text below to add your answers.

### <span style="color:green;font-weight:bold">Need some help?</span>
>If you have any questions about interpreting the plots, get in touch with a demonstrator (in the room or online) and show them your screen to talk through what you're thinking.

### Your answers

**Question 1**: 

**Question 2**:

**Question 3**:

## Submit your work

1. Ensure you have added answers to all the questions and save your file (in the menu bar, click File > Save Notebook).

2. In the file browser, right-click the `prac2_logging_site_monitoring.ipynb` file, and rename it to `prac2_logging_site_monitoring_<FirstName>_<LastName>.ipynb`.

3. In the file browser, right-click the renamed file and press "Download".